In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [2]:
df= pd.read_csv("../data/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.isnull().sum()
y= df.Survived
x= df.drop(['PassengerId','Name','Ticket','Cabin','Survived'],axis= 1)
x_train,x_test,y_train,y_test= train_test_split(x,y, test_size=0.2, random_state= 0)

In [4]:
categorical_cols = [cname for cname in x_train if x_train[cname].nunique()< 10 and x_train[cname].dtype == 'object']
numerical_cols = [cname for cname in x_train if x_train[cname].dtype in ['int64','float64']]
numerical_transformer= SimpleImputer(strategy='median')
categorical_tranformer= Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_tranformer, categorical_cols)
    ])


In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [6]:
model =XGBClassifier(n_estimators= 100, learning_rate= 0.1, max_depth= 5)

In [ ]:
xgboostpipeline = Pipeline(steps= [('prepro', preprocessor),
                               ('model', model)])
xgboostpipeline.fit(x_train,y_train)
df_valid= pd.read_csv('../data/test.csv')
df_valid.head()
preds = xgboostpipeline.predict(x_test)
accuracy1 = accuracy_score(y_test, preds)
print(accuracy1)


0.8435754189944135


In [ ]:
from sklearn.metrics import confusion_matrix

cm1 = confusion_matrix(y_test, preds)

print(cm1)

[[102   8]
 [ 20  49]]


In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.84      0.93      0.88       110
           1       0.86      0.71      0.78        69

    accuracy                           0.84       179
   macro avg       0.85      0.82      0.83       179
weighted avg       0.85      0.84      0.84       179



In [14]:
from sklearn.linear_model import LogisticRegression

In [20]:
modelog = LogisticRegression(random_state=0, max_iter=1000)
logisticregpipeline = Pipeline(steps=[
    ('prepro', preprocessor),
    ('modelo',modelog)
])
logisticregpipeline.fit(x_train, y_train)
preds = logisticregpipeline.predict(x_test)
accuracy2 = accuracy_score(y_test, preds)
print(accuracy2)
cm2 = confusion_matrix(y_test, preds)
print(cm2)
print(classification_report(y_test, preds))



0.7988826815642458
[[92 18]
 [18 51]]
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       110
           1       0.74      0.74      0.74        69

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [21]:
from sklearn.ensemble import RandomForestClassifier
modelrandom = RandomForestClassifier(max_depth=8, n_estimators=200, random_state=0)
randompipeline = Pipeline(steps=[
    ('prepro',preprocessor),
    ('moder',modelrandom)
])
randompipeline.fit(x_train,y_train)
preds = randompipeline.predict(x_test)
accuracy3 = accuracy_score(y_test, preds)
print(accuracy3)
cm3 = confusion_matrix(y_test, preds)
print(cm3)
print(classification_report(y_test, preds))


0.8491620111731844
[[103   7]
 [ 20  49]]
              precision    recall  f1-score   support

           0       0.84      0.94      0.88       110
           1       0.88      0.71      0.78        69

    accuracy                           0.85       179
   macro avg       0.86      0.82      0.83       179
weighted avg       0.85      0.85      0.85       179

